In [1]:
from pathlib import Path

import util
from util import workflow

browser = False
file = util.notebook_file() if util.is_notebook() else __file__
tag = util.file_tag(file)
root_path = Path("..")
data_path = util.data_path(root_path)

In [2]:
# Build
from automol.graph import enum

import automech
from automech.schema import Species

mech0 = automech.io.read(data_path / "full_raw.json")
mech = automech.from_smiles(spc_smis=["C1=CCCC1", "C12C(O2)CCC1"], src_mech=mech0)
#  - add HO2 addition to *ene*
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.pi2_addition,
    rcts_=[None, "O[O]"],
    spc_col_=Species.smiles,
    src_mech=mech0,
)
#  - add ring-forming scission
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.ring_forming_scission,
    src_mech=mech0,
)
#  - enumerate HO2 abstractions from *ene* and *1-2epoxy*
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.abstraction,
    rcts_=[["C1=CCCC1", "C12C(O2)CCC1"], "O[O]"],
    spc_col_=Species.smiles,
    src_mech=mech0,
)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

In [3]:
# Write
workflow.write(mech=mech, tag=tag, root_path=root_path, browser=browser)


Finalizing build for...
Mechanism(
  reactions=DataFrame(
    shape: (8, 5)
    ┌────────────────┬────────────────┬───────────┬──────────────────────────┬─────────────────────────┐
    │ reactants      ┆ products       ┆ formula   ┆ rate                     ┆ colliders               │
    │ ---            ┆ ---            ┆ ---       ┆ ---                      ┆ ---                     │
    │ list[str]      ┆ list[str]      ┆ struct[3] ┆ struct[7]                ┆ struct[11]              │
    ╞════════════════╪════════════════╪═══════════╪══════════════════════════╪═════════════════════════╡
    │ ["S(722)"]     ┆ ["C5H8(522)",  ┆ {5,9,2}   ┆ {{1.0,0.0,0.0},true,"Plo ┆ {null,null,null,null,nu │
    │                ┆ "HO2(8)"]      ┆           ┆ g",nul…                  ┆ ll,null…                │
    │ ["S(722)"]     ┆ ["C5H8O(825)", ┆ {5,9,2}   ┆ {{1.0,0.0,0.0},true,"Plo ┆ {null,null,null,null,nu │
    │                ┆ "OH(4)"]       ┆           ┆ g",nul…                  ┆ ll,n

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]


Writing mechanism...
../data/B_rh-ho2_p1v1_raw.json
../data/B_rh-ho2_p1v1.json
../data/mechanalyzer/B_rh-ho2_p1v1.dat
../data/mechanalyzer/B_rh-ho2_p1v1.csv


  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]


Stereoexpansion errors:


In [4]:
# Read
workflow.read(tag=tag, root_path=root_path)


Reading mechanisms...

Adding calculated thermo...

Adding calculated rates...

Expanding and updating parent...


  0%|          | 0/3151 [00:00<?, ?it/s]


Writing mechanism...
../data/B_rh-ho2_p1v1_calc.json
../data/full_B_rh-ho2_p1v1_control.json
../data/full_B_rh-ho2_p1v1_calc.json
../data/chemkin/full_B_rh-ho2_p1v1_control.dat
../data/chemkin/full_B_rh-ho2_p1v1_calc.dat

Compare calculated mechanism to parent mechanism...

1. Original (compare):
S(722) = C5H8O(825) + OH(4)                            1.000E+00      0.000      0.000
    PLOG  /                                 0.0009870  9.620E+01      2.050      3.050/
    PLOG  /                                 0.0009870  4.840E+35     -7.840      19.69/
    PLOG  /                                  0.009870  1.290E+13     -1.370      5.946/
    PLOG  /                                  0.009870  2.230E+38     -8.490      20.33/
    PLOG  /                                   0.09870  3.220E+22     -4.170      8.727/
    PLOG  /                                   0.09870  2.120E+32     -6.320      18.79/
    PLOG  /                                    0.9870  1.280E+25     -4.670      10.25

In [5]:
# # Simulate
# workflow.simulate(full_tag=f"full_{tag}_calc", root_path=root_path)
# workflow.simulate(full_tag=f"full_{tag}_control", root_path=root_path)